In [ ]:
FILE_PATH = "1_rawdata.csv"

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Column:
    Timestamp = "timestamp"
    # Raw data
    LeftX = "left_x"
    LeftY = "left_y"
    RightX = "right_x"
    RightY = "right_y"
    LeftDiameter = "left_diameter"
    RightDiameter = "right_diameter"

    # Refined data
    EyeX = "eye_x"
    EyeY = "eye_y"
    Diameter = "diameter"

    # Fixation
    StartTimestamp = "start_timestamp"
    EndTimestamp = "end_timestamp"
    Duration = "duration"

In [ ]:
import polars as pl
from pathlib import Path

def read_raw_data(file_path: Path) -> pl.DataFrame:
    schema = {
        "Time": pl.UInt64,
        "L Raw X [px]": pl.Float64,
        "L Raw Y [px]": pl.Float64,
        "R Raw X [px]": pl.Float64,
        "R Raw Y [px]": pl.Float64,
        "L Mapped Diameter [mm]": pl.Float64,
        "R Mapped Diameter [mm]": pl.Float64
    }

    mapping = {
        "Time": Column.Timestamp,
        "L Raw X [px]": Column.LeftX,
        "L Raw Y [px]": Column.LeftY,
        "R Raw X [px]": Column.RightX,
        "R Raw Y [px]": Column.RightY,
        "L Mapped Diameter [mm]": Column.LeftDiameter,
        "R Mapped Diameter [mm]": Column.RightDiameter
    }

    df = pl.read_csv(file_path, columns=list(mapping.keys()), schema_overrides=schema).rename(mapping)
    return df


df = read_raw_data(FILE_PATH)
df

In [ ]:
import polars as pl

def _fuse_eyes(left_column: str, right_column: str, target_column: str) -> pl.Series:
    left_data, right_data = pl.col(left_column), pl.col(right_column)

    return (
        pl.when((left_data > 0) & (right_data > 0)).then((left_data + right_data) / 2.0)
        .when(left_data > 0).then(left_data)
        .otherwise(right_data)
        .alias(target_column)
    )

def fuse_raw_data(df: pl.DataFrame) -> pl.DataFrame:
    transformed_df = (
        df
        # Filter out rows where both eyes are missing (0.0)
        .filter(
            ((pl.col(Column.LeftX) > 0) | (pl.col(Column.RightX) > 0))
            & ((pl.col(Column.LeftY) > 0) | (pl.col(Column.RightY) > 0))
            & ((pl.col(Column.LeftDiameter) > 0) | (pl.col(Column.RightDiameter) > 0))
        )
        # Apply the fusion logic to each pair
        .select([
            Column.Timestamp,
            _fuse_eyes(Column.LeftX, Column.RightX, Column.EyeX),
            _fuse_eyes(Column.LeftY, Column.RightY, Column.EyeY),
            _fuse_eyes(Column.LeftDiameter, Column.RightDiameter, Column.Diameter)
        ])
    )
    return transformed_df



transformed_df = fuse_raw_data(df)
transformed_df

In [ ]:
import polars as pl
import numpy as np

def extract_fixations_by_idt_algorithm(
    df: pl.DataFrame,
    t1: float,
    t2: float,
    min_duration: float
) -> pl.DataFrame:
    """
    Pure Polars implementation of Dispersion-based fixation detection.
    """
    t2_sq = t2**2

    return (
        df.lazy()
        # 1. Rolling Dispersion (t1) - Identify potential fixation samples
        .with_columns([
            (
                (pl.col(Column.EyeX).rolling_max(5) - pl.col(Column.EyeX).rolling_min(5)).abs() +
                (pl.col(Column.EyeY).rolling_max(5) - pl.col(Column.EyeY).rolling_min(5)).abs()
            ).alias("dispersion")
        ])
        .with_columns([
            (pl.col("dispersion") < t1).alias("is_still")
        ])
        # 2. Grouping: Create unique IDs for continuous blocks of "is_still"
        .with_columns([
            (pl.col("is_still") != pl.col("is_still").shift()).fill_null(True).cum_sum().alias("run_id")
        ])
        .filter(pl.col("is_still"))
        # 3. t2 Refinement: Calculate distance from centroid within each run_id
        # We calculate means per group and join them back to the rows for a vectorized distance check
        .with_columns([
            pl.col(Column.EyeX).mean().over("run_id").alias("_mean_x"),
            pl.col(Column.EyeY).mean().over("run_id").alias("_mean_y"),
        ])
        .filter(
            ((pl.col(Column.EyeX) - pl.col("_mean_x")).pow(2) +
             (pl.col(Column.EyeY) - pl.col("_mean_y")).pow(2)) < t2_sq
        )
        # 4. Final Aggregation: Group by the run_id to get the final fixation statistics
        .group_by("run_id")
        .agg([
            pl.col(Column.Timestamp).min().alias(Column.StartTimestamp),
            pl.col(Column.Timestamp).max().alias(Column.EndTimestamp),
            pl.col(Column.EyeX).mean().alias(Column.EyeX),
            pl.col(Column.EyeY).mean().alias(Column.EyeY),
            pl.col(Column.Diameter).mean().alias(Column.Diameter),
            (pl.col(Column.Timestamp).max() - pl.col(Column.Timestamp).min()).alias(Column.Duration),
            pl.len().alias("sample_count")
        ])
        # Final filter for duration
        .filter(pl.col(Column.Duration) >= min_duration)
        .drop("run_id")
        .collect()
    )

# usage
# fixations = extract_fixations_pure_polars(transformed_df, 25.0, 150000)

# Execute
fixations_df = extract_fixations_by_idt_algorithm(transformed_df, 25.0, 0.1, 150.0)
fixations_df

In [ ]:
import matplotlib.pyplot as plt
import polars as pl

def plot_fixations(
    raw_df: pl.DataFrame,
    fix_df: pl.DataFrame,
    title: str = "iView X Fixations"
):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.set_title(f"{title} | Fixations count: {len(fix_df)}")
    ax.set_xlabel("Horizontal (px)")
    ax.set_ylabel("Vertical (px)")

    # 1. Plot Raw Gaze (using Column class names)
    # Extracting to numpy is fastest for Matplotlib
    ax.scatter(
        raw_df[Column.EyeX].to_numpy(),
        raw_df[Column.EyeY].to_numpy(),
        color='blue', marker='+', alpha=0.3, label='Raw Gaze'
    )

    if len(fix_df) > 0:
        # 2. Plot Fixation Centers
        ax.scatter(
            fix_df[Column.EyeX].to_numpy(),
            fix_df[Column.EyeY].to_numpy(),
            color='red', marker='o', s=50, edgecolors='black', label='Fixations'
        )

        # 3. Annotate durations
        # Efficient iteration using zip() instead of iterrows
        coords_and_dur = zip(
            fix_df[Column.EyeX],
            fix_df[Column.EyeY],
            fix_df[Column.Duration]
        )

        for x, y, dur in coords_and_dur:
            ax.text(x, y, f"{dur:.0f}", color='darkred', fontsize=10, weight='bold')

    ax.legend(loc='best')
    plt.show()

    ## 4. Efficient Console Output
    #if len(fix_df) > 0:
    #    print(f"\n--- Fixations Summary ({len(fix_df)}) ---")
    #    # Converting to dict of lists is much faster than row-by-row logic
    #    data = fix_df.to_dicts()
    #    for i, row in enumerate(data, 1):
    #        print(
    #            f"{i}. x={row[Column.EyeX]:.3f}, y={row[Column.EyeY]:.3f}, "
    #            f"{row[Column.Duration]:.0f} units "
    #            f"[{row[Column.StartTimestamp]}–{row[Column.EndTimestamp]}]"
    #        )
    #else:
    #    print("--- No fixations detected ---")

# Usage:
plot_fixations(transformed_df, fixations_df)

In [ ]:
import polars as pl
from pathlib import Path

def read_txt_raw_data(file_path: Path) -> pl.DataFrame:
    new_columns = [Column.EyeX, Column.EyeY, Column.Timestamp]

    return (
        pl.read_csv(
            file_path,
            has_header=False,
            separator=" ",
            new_columns=new_columns,
            truncate_ragged_lines=True,
            schema_overrides={
                "column_1": pl.Float64,
                "column_2": pl.Float64,
                "column_3": pl.Float64,
            }
        )
        .with_columns([
            pl.col(Column.Timestamp).cast(pl.UInt64),
            pl.lit(0.2).alias(Column.Diameter)
        ])
        .select([Column.Timestamp, Column.EyeX, Column.EyeY, Column.Diameter])
    )

# Usage
demo_df = read_txt_raw_data(Path("demo_data.txt"))
demo_df

In [ ]:
demo_fixations_df = extract_fixations_by_idt_algorithm(demo_df, 0.25, 0.1, 150)
demo_fixations_df

In [ ]:
plot_fixations(demo_df, demo_fixations_df)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

def plot_eye_metrics(df: pl.DataFrame):
    # 1. Calculate Velocity (angular or pixel displacement)
    # Velocity is the Euclidean distance between consecutive gaze points
    df_with_vel = df.with_columns([
        (
            ((pl.col(Column.EyeX) - pl.col(Column.EyeX).shift(1)).pow(2) +
             (pl.col(Column.EyeY) - pl.col(Column.EyeY).shift(1)).pow(2)).sqrt() /
            (pl.col(Column.Timestamp) - pl.col(Column.Timestamp).shift(1))
        ).fill_null(0).alias("velocity")
    ])

    # Convert to numpy for plotting
    time = df_with_vel[Column.Timestamp].to_numpy()
    dilation = df_with_vel[Column.Diameter].to_numpy()
    velocity = df_with_vel["velocity"].to_numpy()

    # 2. Setup Subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True,
                                   gridspec_kw={'height_ratios': [1, 2]})
    plt.subplots_adjust(hspace=0.05)

    # Top Plot: Pupil Dilation
    ax1.plot(time, dilation, color='blue', linewidth=0.8, label='Pupil Dilation')
    ax1.set_ylabel("Dilation (mm)")
    ax1.set_ylim(dilation.min() * 0.9, dilation.max() * 1.1)
    ax1.grid(True, alpha=0.2)
    ax1.text(0.01, 0.85, "Pupil dilation", transform=ax1.transAxes, fontweight='bold')

    # Bottom Plot: Velocity (Filtered Data)
    # We mirror the red downward spikes from your reference image
    ax2.plot(time, -velocity, color='red', linewidth=0.5, alpha=0.8)
    ax2.set_ylabel("Velocity")
    ax2.set_xlabel("Timestamp")
    ax2.set_ylim(-velocity.max() * 1.1, 5) # 5 for a bit of headroom
    ax2.grid(True, alpha=0.2)
    ax2.text(0.01, 0.05, "Velocity (Filtered data)", transform=ax2.transAxes, fontweight='bold')

    # Format X-axis to show units clearly
    ax2.ticklabel_format(style='plain', axis='x')

    plt.show()


plot_eye_metrics(transformed_df)

In [ ]:
def plot_eye_metrics_refined(df: pl.DataFrame, velocity_threshold: float = 0.5):
    # 1. Calculate Velocity with better scaling
    # We use .diff() for readability and speed
    df_with_vel = df.with_columns([
        (
            ((pl.col(Column.EyeX).diff().pow(2) +
              pl.col(Column.EyeY).diff().pow(2)).sqrt()) /
            (pl.col(Column.Timestamp).diff().fill_null(1))
        ).fill_null(0).alias("velocity")
    ])

    # 2. Extract data
    time = df_with_vel[Column.Timestamp].to_numpy()
    dilation = df_with_vel[Column.Diameter].to_numpy()
    velocity = df_with_vel["velocity"].to_numpy()

    # 3. Setup Subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                   gridspec_kw={'height_ratios': [1, 1]})
    plt.subplots_adjust(hspace=0.1)

    # Top Plot: Pupil Dilation
    # Masking zeros/blinks for a cleaner line
    dilation_masked = np.where(dilation == 0, np.nan, dilation)
    ax1.plot(time, dilation_masked, color='blue', linewidth=1)
    ax1.set_ylabel("Dilation (mm)")
    ax1.grid(True, linestyle='--', alpha=0.5)
    ax1.set_title("Pupil Dilation & Saccadic Velocity", loc='left', fontweight='bold')

    # Bottom Plot: Velocity with Threshold
    # Mirroring the reference: spikes go DOWN from 0
    ax2.plot(time, -velocity, color='red', linewidth=0.7, alpha=0.7)

    # Add the Threshold Line
    ax2.axhline(-velocity_threshold, color='black', linestyle=':', label='Saccade Threshold')

    # Fill area beyond threshold to highlight saccades (High-perf visualization)
    ax2.fill_between(time, -velocity, -velocity_threshold,
                     where=(velocity > velocity_threshold),
                     color='red', alpha=0.2)

    ax2.set_ylabel("Velocity (px/unit time)")
    ax2.set_xlabel("Timestamp")

    # Tighten the Y-limit to focus on the data spikes, not the empty space
    ax2.set_ylim(-max(velocity.max(), velocity_threshold * 2), 0.1)
    ax2.grid(True, linestyle='--', alpha=0.5)
    ax2.legend(loc='lower right')

    plt.show()

plot_eye_metrics_refined(transformed_df)

In [ ]:
import polars as pl
import numpy as np

def plot_eye_metrics_grafix_style(df: pl.DataFrame, velocity_threshold: float = 0.5):
    # 1. PRE-PROCESSING: Smooth and Calculate
    df_vel = (
        df.lazy()
        .with_columns([
            pl.col(Column.EyeX).rolling_mean(window_size=5, center=True).alias("x_s"),
            pl.col(Column.EyeY).rolling_mean(window_size=5, center=True).alias("y_s")
        ])
        .with_columns([
            (
                ((pl.col("x_s").diff().pow(2) + pl.col("y_s").diff().pow(2)).sqrt()) /
                (pl.col(Column.Timestamp).diff().cast(pl.Float64).fill_null(1.0))
            ).fill_null(0.0).alias("raw_v")
        ])
        .with_columns([
            # Normalize by max to keep scale between 0 and 1, avoiding Inf/NaN
            (pl.col("raw_v") / (pl.col("raw_v").max() + 1e-9)).alias("norm_velocity")
        ])
        .collect()
    )

    # 2. Extract and sanitize for Numpy/Matplotlib
    time = df_vel[Column.Timestamp].to_numpy()
    dilation = df_vel[Column.Diameter].to_numpy()
    velocity = df_vel["norm_velocity"].to_numpy()

    # Determine a safe Y-limit
    v_max = np.nanmax(velocity) if len(velocity) > 0 else 1.0
    # Ensure v_max is finite and at least the threshold
    safe_limit = max(float(v_max), velocity_threshold * 2, 1.0)

    # 3. PLOTTING
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    # Top Plot: Pupil
    ax1.plot(time, np.where(dilation == 0, np.nan, dilation), color='#1f77b4', lw=1.5)
    ax1.set_ylabel("Pupil Diameter (mm)")

    # Bottom Plot: Velocity
    ax2.plot(time, -velocity, color='#d62728', lw=0.8)
    ax2.axhline(-velocity_threshold, color='black', ls='--', label='Saccade Threshold')

    # Highlight saccades
    ax2.fill_between(time, -velocity, -velocity_threshold,
                     where=(velocity > velocity_threshold),
                     color='red', alpha=0.3)

    # The Fix: Use the safe_limit
    ax2.set_ylim(-safe_limit, 0)
    ax2.set_ylabel("Normalized Velocity")
    ax2.legend()

    plt.show()

plot_eye_metrics_grafix_style(transformed_df)

In [ ]:
plot_eye_metrics_grafix_style(demo_df)

In [ ]:
from IPython.display import display, clear_output
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

def plot_interactive_eye_metrics(df: pl.DataFrame):
    # 1. Pre-calculate once (Super Effective)
    processed_df = (
        df.lazy()
        .with_columns([
            pl.col(Column.EyeX).rolling_mean(5, center=True).alias("x_s"),
            pl.col(Column.EyeY).rolling_mean(5, center=True).alias("y_s")
        ])
        .with_columns([
            (
                ((pl.col("x_s").diff().pow(2) + pl.col("y_s").diff().pow(2)).sqrt()) /
                (pl.col(Column.Timestamp).diff().cast(pl.Float64).fill_null(1.0))
            ).fill_null(0.0).alias("raw_v")
        ])
        .with_columns([
            (pl.col("raw_v") / (pl.col("raw_v").max() + 1e-9)).alias("norm_v")
        ])
        .collect()
    )

    time = processed_df[Column.Timestamp].to_numpy()
    dilation = processed_df[Column.Diameter].to_numpy()
    velocity = processed_df["norm_v"].to_numpy()

    # 2. Setup the UI output container
    output = widgets.Output()

    def update(change):
        threshold = change['new']
        with output:
            clear_output(wait=True) # wait=True prevents "flicker"

            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
            plt.subplots_adjust(hspace=0.1)

            # Pupil Plot
            ax1.plot(time, np.where(dilation == 0, np.nan, dilation), color='#1f77b4', lw=1.2)
            ax1.set_ylabel("Pupil (mm)")

            # Velocity Plot
            ax2.plot(time, -velocity, color='#d62728', lw=0.7, alpha=0.5)
            ax2.axhline(-threshold, color='black', ls='--', lw=1.5)
            ax2.fill_between(time, -velocity, -threshold,
                             where=(velocity > threshold),
                             color='red', alpha=0.3)

            ax2.set_ylim(-1.1, 0.05)
            ax2.set_ylabel("Norm. Velocity")

            # Count saccades
            saccades = np.sum((velocity[:-1] < threshold) & (velocity[1:] >= threshold))
            ax1.set_title(f"Saccade Threshold: {threshold:.2f} | Number of saccades: {saccades}",
                          loc='left', fontweight='bold')

            plt.show()

    # 3. Create Slider and attach the observer
    slider = widgets.FloatSlider(
        value=0.5, min=0.01, max=1.0, step=0.01,
        description='Threshold:', continuous_update=False # False = update on release
    )
    slider.observe(update, names='value')

    # Display the UI
    display(slider, output)

    # Trigger initial plot
    update({'new': 0.5})

plot_interactive_eye_metrics(demo_df)

In [ ]:
plot_interactive_eye_metrics(transformed_df)

In [ ]:
import polars as pl
import numpy as np

def extract_fixations_IVT(
    df: pl.DataFrame,
    velocity_threshold: float,
    min_duration: float
) -> pl.DataFrame:
    """
    I-VT (Velocity-Threshold) Implementation.
    """
    return (
        df.lazy()
        # 1. Calculate Velocity (px or units per ms)
        .with_columns([
            (
                ((pl.col(Column.EyeX).diff().pow(2) +
                  pl.col(Column.EyeY).diff().pow(2)).sqrt()) /
                (pl.col(Column.Timestamp).diff().fill_null(1))
            ).alias("velocity")
        ])
        # 2. Classify points
        .with_columns([
            (pl.col("velocity") < velocity_threshold).alias("is_fixation")
        ])
        # 3. Create Event IDs (grouping consecutive fixation points)
        .with_columns([
            (pl.col("is_fixation") != pl.col("is_fixation").shift())
            .fill_null(True)
            .cum_sum()
            .alias("event_id")
        ])
        # 4. Aggregate Fixations
        .filter(pl.col("is_fixation"))
        .group_by("event_id")
        .agg([
            pl.col(Column.Timestamp).min().alias(Column.StartTimestamp),
            pl.col(Column.Timestamp).max().alias(Column.EndTimestamp),
            pl.col(Column.EyeX).mean().alias(Column.EyeX),
            pl.col(Column.EyeY).mean().alias(Column.EyeY),
            pl.col(Column.Diameter).mean().alias(Column.Diameter),
            (pl.col(Column.Timestamp).max() - pl.col(Column.Timestamp).min()).alias(Column.Duration)
        ])
        # 5. Temporal Filter
        .filter(pl.col(Column.Duration) >= min_duration)
        .drop("event_id")
        .collect()
    )

ivt_fixations = extract_fixations_IVT(transformed_df, 0.05, 100)
ivt_fixations

In [ ]:
plot_fixations(transformed_df, ivt_fixations)

In [ ]:
plot_fixations(demo_df, extract_fixations_IVT(demo_df, 0.005, 100))

In [ ]:
# I-HMM (Hidden Markov Model Identification)
# I-MST (Minimum Spanning Tree Identification)
# I-VMP (Velocity-MST-Phases)

# Savitzky-Golay (SG)
# Infinite impulse response (IIR)
# Finite impulse response (FIR)

In [ ]:
# https://www.kaggle.com/datasets/namratasri01/eye-movement-data-set-for-desktop-activities
# - example data

In [ ]:
# https://ceur-ws.org/Vol-3710/paper6.pdf
# - great step-by-step description of data cleanup pipeline
# https://www.diva-portal.org/smash/get/diva2:573446/FULLTEXT01.pdf
# - detailed work about algorithms, formulas and real-time filtering as well
# https://www.researchgate.net/publication/375428985_Handling_Noisy_Data_in_Eye-Tracking_Research_Methods_and_Best_Practices
# - noise-reduction techniques
# https://www.researchgate.net/publication/374853132_Filtering_eye-tracking_data_from_an_EyeLink_1000_Comparing_heuristic_savitzky-golay_IIR_and_FIR_digital_filters
# filtering techniques

In [ ]:
# https://www.sciencedirect.com/science/article/abs/pii/S0957417420308071
# - Using ML
# https://www.frontiersin.org/journals/neuroscience/articles/10.3389/fnins.2021.578439/full
# - small screen, e.g. mobile devices